In [1]:
!pip3 install folium wget pandas
import folium
import wget
import pandas as pd
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=840098e32eb9be30da2d133ff819f4c2c2f9957bbba20374153d5ab9553daf6a
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = pd.read_csv(spacex_csv_file)

In [3]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [12]:
nasa_coordinate = [28.562302, -80.577356]

site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for index, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    # Create a Circle for each launch site with its name as a popup
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(site['Launch Site']))

    # Create a Marker for each launch site with its name as a DivIcon label
    marker = folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site'],
            )
        )

    # Add both the Circle and Marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map

In [13]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

In [14]:
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    # build coordinate from record['Lat'], record['Long']
    coordinate = [record['Lat'], record['Long']]

    # create folium.Marker with icon=folium.Icon(color='white', icon_color=record['marker_color'])
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=record['Launch Site'] # add popup with the Launch Site name
    )
    # add to marker_cluster
    marker_cluster.add_child(marker)
site_map

In [15]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [16]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [17]:
launch_site_lat, launch_site_lon = 28.56230197, -80.57735648   # CCAFS SLC-40
coastline_lat, coastline_lon = 28.56423, -80.56812            # YOUR hovered value

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Coastline distance: {distance_coastline:.2f} km")

# Distance marker
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline)))
site_map.add_child(distance_marker)

# PolyLine site → coastline
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon],[coastline_lat, coastline_lon]], weight=1)
site_map.add_child(lines)
site_map

Coastline distance: 0.93 km


In [18]:
proximities = {
    'Railway': (28.56347, -80.57709),   # ← the ✕-dashed line near the pad
    'Highway': (28.56368, -80.57083),   # ← Samuel C Phillips Parkway
    'City':    (28.38458, -80.60654)    # ← e.g., Cape Canaveral/Cocoa (zoom out!)
}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    print(f"{name}: {d:.2f} km")
    site_map.add_child(folium.Marker([lat, lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#0000ff;"><b>%s %.2f KM</b></div>' % (name, d))))
    site_map.add_child(folium.PolyLine(locations=[[launch_site_lat, launch_site_lon],[lat, lon]], weight=1, color='blue'))
site_map

Railway: 0.13 km
Highway: 0.66 km
City: 19.97 km


In [19]:
site_map.save('site_map.html')